In [1]:
import sys
sys.executable

'c:\\Users\\betsie_0410\\AppData\\Local\\anaconda3\\envs\\energiaenv\\python.exe'

In [2]:
import mpi4py
import mpisppy
import mpisppy.utils.sputils as sputils
from mpisppy.opt.ef import ExtensiveForm
print("All imports OK")

[    0.00] Initializing mpi-sppy
All imports OK


In [3]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [4]:
import os
os.environ["GRB_LICENSE_FILE"] = r"C:\Users\betsie_0410\gurobi.lic"

In [5]:
from pyomo.environ import *
import mpisppy.utils.sputils as sputils
from mpisppy.opt.lshaped import LShapedMethod
from pyomo.environ import value as pyoval
import matplotlib.pyplot as plt
from matplotlib import rc

In [12]:
import pyomo.environ as pyo

def build_model(yields):
    model = pyo.ConcreteModel()

    # Variables
    model.X = pyo.Var(["WHEAT", "CORN", "BEETS"], within=pyo.NonNegativeReals)
    model.Y = pyo.Var(["WHEAT", "CORN"], within=pyo.NonNegativeReals)
    model.W = pyo.Var(
        ["WHEAT", "CORN", "BEETS_FAVORABLE", "BEETS_UNFAVORABLE"],
        within=pyo.NonNegativeReals,
    )

    # Objective function
    model.PLANTING_COST = 150 * model.X["WHEAT"] + 230 * model.X["CORN"] + 260 * model.X["BEETS"]
    model.PURCHASE_COST = 238 * model.Y["WHEAT"] + 210 * model.Y["CORN"]
    model.SALES_REVENUE = (
        170 * model.W["WHEAT"] + 150 * model.W["CORN"]
        + 36 * model.W["BEETS_FAVORABLE"] + 10 * model.W["BEETS_UNFAVORABLE"]
    )
    model.OBJ = pyo.Objective(
        expr=model.PLANTING_COST + model.PURCHASE_COST - model.SALES_REVENUE,
        sense=pyo.minimize
    )

    # Constraints
    model.CONSTR= pyo.ConstraintList()

    model.CONSTR.add(pyo.summation(model.X) <= 500)
    model.CONSTR.add(
        yields[0] * model.X["WHEAT"] + model.Y["WHEAT"] - model.W["WHEAT"] >= 200
    )
    model.CONSTR.add(
        yields[1] * model.X["CORN"] + model.Y["CORN"] - model.W["CORN"] >= 240
    )
    model.CONSTR.add(
        yields[2] * model.X["BEETS"] - model.W["BEETS_FAVORABLE"] - model.W["BEETS_UNFAVORABLE"] >= 0
    )
    model.W["BEETS_FAVORABLE"].setub(6000)

    return model

In [7]:
yields = [2.5, 3, 20]
model = build_model(yields)
solver = pyo.SolverFactory("gurobi")
solver.solve(model)

# Display the objective value to one decimal place
print(f"{pyo.value(model.OBJ):.1f}")

-118600.0


In [8]:
import sys
print(sys.executable)


c:\Users\betsie_0410\AppData\Local\anaconda3\envs\energiaenv\python.exe


In [9]:
import mpisppy.utils.sputils as sputils

def scenario_creator(scenario_name):
    if scenario_name == "good":
        yields = [3, 3.6, 24]
    elif scenario_name == "average":
        yields = [2.5, 3, 20]
    elif scenario_name == "bad":
        yields = [2, 2.4, 16]
    else:
        raise ValueError("Unrecognized scenario name")

    model = build_model(yields)
    sputils.attach_root_node(model, model.PLANTING_COST, [model.X])
    model._mpisppy_probability = 1.0 / 3
    return model

In [13]:
from mpisppy.opt.lshaped import LShapedMethod

all_scenario_names = ["good", "average", "bad"]
bounds = {name: -432000 for name in all_scenario_names}
options = {
    "root_solver": "gurobi",
    "sp_solver": "gurobi",
    "sp_solver_options" : {"threads" : 1},
    "valid_eta_lb": bounds,
    "max_iter": 10,
}

ls = LShapedMethod(options, all_scenario_names, scenario_creator)
result = ls.lshaped_algorithm()

variables = ls.gather_var_values_to_rank0()
for ((scen_name, var_name), var_value) in variables.items():
    print(scen_name, var_name, var_value)

[  264.51] Initializing SPBase
Current Iteration: 1 Time Elapsed:    0.00 Current Objective: -Inf
Current Iteration: 2 Time Elapsed:    0.35 Time Spent on Last Master:    0.09 Time Spent Generating Last Cut Set:    0.27 Current Objective: -1296000.00
Current Iteration: 3 Time Elapsed:    0.71 Time Spent on Last Master:    0.09 Time Spent Generating Last Cut Set:    0.27 Current Objective: -160000.00
Current Iteration: 4 Time Elapsed:    1.06 Time Spent on Last Master:    0.09 Time Spent Generating Last Cut Set:    0.26 Current Objective: -113750.00
Converged in 4 iterations.
Total Time Elapsed:    1.41 Time Spent on Last Master:    0.09 Time spent verifying second stage:    0.25 Final Objective: -108390.00
good X[BEETS] 250.00000000000023
good X[CORN] 79.99999999999964
good X[WHEAT] 170.0000000000001
average X[BEETS] 250.00000000000023
average X[CORN] 79.99999999999964
average X[WHEAT] 170.0000000000001
bad X[BEETS] 250.00000000000023
bad X[CORN] 79.99999999999964
bad X[WHEAT] 170.0000